# 02 – Rozdelenie podľa produktu a baseline (TF-IDF + logistická regresia)

- čistenie: prázdne texty, duplicity v pravých recenziách, len pravé 5/5 recenzie
- kontrola metadátových artefaktov (stĺpce, ktoré odlišujú spam bez použitia textu)
- rozdelenie train/val/test podľa produktu
- baseline len z textu (`reviewText`), evaluácia každý generátor proti každému (train na A, test na B)
- test artefaktu z dĺžky (rozhodovací strom)

Vstup modelu j iba text.

In [1]:
from pathlib import Path

DATA = Path("../../data/FraudSquad-Data/Amazon")
REAL_PATH = DATA / "amazon_real.csv"
SPAM_PATHS = {
    "llama3":    DATA / "reviews_from_llama3_8b.csv",
    "qwen_72b":  DATA / "reviews_from_qwen_72b.csv",
    "qwen_dsr1": DATA / "reviews_from_qwen_dsr1.csv",
}
COL = {"text": "reviewText", "rating": "star", "product": "product", "user": "user", "time": "reviewTime"}

SEED = 42
VAL_FRAC = 0.15
TEST_FRAC = 0.15
DEDUP_REAL = True        # vyhodiť duplicitné texty v pravých recenziách
REAL_5STAR_ONLY = True   # spam je vždy 5★ -> porovnanie len s pravými 5★
DROP_NO_TEXT = True      # vyhodiť recenzie bez písmen/číslic (len emoji, interpunkcia)
MAX_EMOJI = None         # napr. 10: vyhodiť recenzie s viac ako 10 emoji; None = nevyhadzovať
EMOJI_MANY = 5           # hranica pre "veľa emoji" (len pre výpis)

In [2]:
import numpy as np
import pandas as pd

real = pd.read_csv(REAL_PATH)
real_raw = real          # pôvodné pravé recenzie pred čistením (na porovnanie)
spam = {name: pd.read_csv(p) for name, p in SPAM_PATHS.items()}
txt, rat, prod = COL["text"], COL["rating"], COL["product"]
print("real:", real.shape, {k: v.shape for k, v in spam.items()})

real: (86758, 17) {'llama3': (2489, 18), 'qwen_72b': (2495, 18), 'qwen_dsr1': (2500, 18)}


## 1. Čistenie
- odstránenie prázdnych `reviewText`
- deduplikácia pravých recenzií podľa textu (`DEDUP_REAL`)
- len pravé 5★ recenzie (`REAL_5STAR_ONLY`)

In [3]:
n0 = len(real)
real = real.dropna(subset=[txt])
n1 = len(real)
if DEDUP_REAL:
    real = real.drop_duplicates(subset=[txt])
n2 = len(real)
if REAL_5STAR_ONLY:
    real = real[real[rat] >= 5]
n3 = len(real)
print(f"pôvodne {n0} -> bez prázdnych {n1} -> bez duplicít {n2} -> len 5★ {n3}")

for name in spam:
    spam[name] = spam[name].dropna(subset=[txt])
    print(name, len(spam[name]))

pôvodne 86758 -> bez prázdnych 86697 -> bez duplicít 79527 -> len 5★ 55586
llama3 2489
qwen_72b 2495
qwen_dsr1 2500


## 2. Metadátové artefakty
- porovnanie pravých 5★ a spamu: `verified_purchase`, `helpful_vote`, obrázky, `reviewSummary`, konzistencia `product_star` so `star` a `reviewDay` s `reviewTime`
- stĺpce s výrazným rozdielom sa nepoužívajú ako vstup modelu (data leakage)

In [4]:
def meta_row(d):
    out = {}
    if "verified_purchase" in d: out["verified_purchase"] = d["verified_purchase"].astype(float).mean()
    if "helpful_vote" in d:      out["helpful_vote>0"] = (d["helpful_vote"] > 0).mean()
    if "images" in d:            out["má obrázky"] = (d["images"].astype(str) != "[]").mean()
    if "reviewSummary" in d:     out["reviewSummary prítomné"] = d["reviewSummary"].notna().mean()
    if "product_star" in d:
        out["product_star sedí so star"] = (d["product_star"] == d[prod].astype(str) + "_" + d[rat].astype(str)).mean()
    if "reviewDay" in d:
        day = pd.to_datetime(d[COL["time"]], unit="s").dt.strftime("%Y-%m-%d")
        out["reviewDay sedí s reviewTime"] = (day == d["reviewDay"]).mean()
    return out

rows = {"real (5★)": meta_row(real)}
for name, d in spam.items():
    rows[name] = meta_row(d)
display(pd.DataFrame(rows).T.round(3))

,verified_purchase,helpful_vote>0,má obrázky,reviewSummary prítomné,product_star sedí so star,reviewDay sedí s reviewTime
real (5★),0.251,0.194,0.267,1.0,1.000,0.557
llama3,0.000,0.000,0.112,0.0,0.274,0.162
qwen_72b,0.000,0.000,0.112,0.0,0.275,0.167
qwen_dsr1,0.000,0.000,0.112,0.0,0.274,0.168


## 2b. Emoji a recenzie bez textu
- výskyt emoji a recenzií bez písmen/číslic v pravých recenziách a v spame
- TF-IDF s predvoleným tokenizérom emoji ignoruje; transformerové modely (DistilBERT) ich spracujú

In [5]:
import re

EMOJI_RE = re.compile("[\U0001F1E6-\U0001F1FF\U0001F300-\U0001FAFF\u2600-\u27BF\u2B00-\u2BFF]")
JOINERS_RE = re.compile("[\uFE0F\u200D]")

def emoji_count(t):
    return len(EMOJI_RE.findall(t))

def has_no_text(t):
    rest = JOINERS_RE.sub("", EMOJI_RE.sub("", t))
    return re.search(r"\w", rest) is None

def emoji_stats(d):
    t = d[txt].fillna("").astype(str)
    n = t.map(emoji_count)
    nt = t.map(has_no_text)
    return {
        "riadkov": len(d),
        "má emoji": (n >= 1).mean(),
        f"emoji >= {EMOJI_MANY}": (n >= EMOJI_MANY).mean(),
        "max emoji v jednej": n.max(),
        "bez písmen/číslic": nt.mean(),
        "len emoji (bez textu)": ((n >= 1) & nt).mean(),
    }

rows = {"real (pôvodne, všetky)": emoji_stats(real_raw), "real (5★, po čistení)": emoji_stats(real)}
for name, d in spam.items():
    rows[name] = emoji_stats(d)
display(pd.DataFrame(rows).T.round(4))

,riadkov,má emoji,emoji >= 5,max emoji v jednej,bez písmen/číslic,len emoji (bez textu)
"real (pôvodne, všetky)",86758.0,0.0194,0.0032,259.0,0.0028,0.0012
"real (5★, po čistení)",55586.0,0.0201,0.0033,259.0,0.0003,0.0003
llama3,2489.0,0.0000,0.0000,0.0,0.0000,0.0000
qwen_72b,2495.0,0.0000,0.0000,0.0,0.0000,0.0000
qwen_dsr1,2500.0,0.0108,0.0000,2.0,0.0000,0.0000


In [6]:
# Ukážky: recenzie bez textu, recenzie s veľa emoji, spam s emoji
t = real_raw[txt].fillna("").astype(str)
n = t.map(emoji_count)
print("pravé: len emoji / bez textu:")
print(t[t.map(has_no_text)].head(8).tolist())
print("\npravé: veľa emoji:")
print(t[n >= EMOJI_MANY].str.slice(0, 120).head(5).tolist())
for name, d in spam.items():
    ts = d[txt].astype(str)
    hit = ts[ts.map(emoji_count) >= 1]
    print(f"\n{name}: recenzií s emoji = {len(hit)}")
    print(hit.str.slice(0, 120).head(3).tolist())

pravé: len emoji / bez textu:
['👍👍', '👏👏', '', '', '', '', '👍🏾', '👍🏾']

pravé: veľa emoji:
["Doesn't work ever!!! Waaaaa😣😣😥😫😦😤😤😤🤕🤕🤕", 'THE MOVIE BETTER BE WORTH IT!!!!!😖😤😤😤😤😤😤😤😤🤕', 'GOOOD GAME OMG OMG I LOVE IT NEW SKINS but.... If you have an home in offline you should in online do that....<br />BUT ', "It's kinda good but....  I NEED TO DELETE ALL OF MY APPS I HATE THAT delete that plz😔😓😭😾👍👎👑🎧", '💘 LOVE IT AND CUTE ANIMALS GOOD FOR ANIMAL LOVERS DOWNLOAD NOW!!!!!🐈🐢🐭🐷🐶🐱🐺🐰🐒🐍🐻🐯🐆🐾🐾🐾🐾🐾']

llama3: recenzií s emoji = 0
[]

qwen_72b: recenzií s emoji = 0
[]

qwen_dsr1: recenzií s emoji = 27
['This is hands down one of the best music games I’ve played. The beat matching is spot-on, and the 3D effects make it fee', 'I’m obsessed with this game! The mix of upbeat music, colorful visuals, and addictive gameplay is irresistible. The abil', 'I can’t stop playing this game! The combination of catchy tunes, vibrant graphics, and smooth gameplay is unbeatable. Th']


In [7]:
# Filtrovanie podľa konfigurácie (DROP_NO_TEXT, MAX_EMOJI)
def apply_text_filters(d, label):
    before = len(d)
    t = d[txt].astype(str)
    if DROP_NO_TEXT:
        d = d[~t.map(has_no_text)]
    if MAX_EMOJI is not None:
        d = d[d[txt].astype(str).map(emoji_count) <= MAX_EMOJI]
    print(f"{label}: {before} -> {len(d)} (vyhodených {before - len(d)})")
    return d

real = apply_text_filters(real, "real")
spam = {k: apply_text_filters(v, k) for k, v in spam.items()}

real: 55586 -> 55567 (vyhodených 19)
llama3: 2489 -> 2489 (vyhodených 0)
qwen_72b: 2495 -> 2495 (vyhodených 0)
qwen_dsr1: 2500 -> 2500 (vyhodených 0)


## 3. Rozdelenie podľa produktu
- produkty so spamom a ostatné produkty sa delia zvlášť v rovnakých pomeroch (dostatok spamu v teste)
- pravé recenzie aj spam jedného produktu sú v tej istej časti; žiadny produkt z train sa nevyskytuje vo val ani v test
- pomer train / val / test: 70 / 15 / 15

In [8]:
rng = np.random.default_rng(SEED)

spam_prods = set().union(*[set(d[prod]) for d in spam.values()])
other_prods = set(real[prod]) - spam_prods
print("produktov so spamom:", len(spam_prods), "| ostatných:", len(other_prods))
print("produkty spoločné všetkým trom generátorom:", len(set.intersection(*[set(d[prod]) for d in spam.values()])))

def split_products(prods):
    prods = np.array(sorted(prods))
    rng.shuffle(prods)
    n = len(prods); nt = int(n * TEST_FRAC); nv = int(n * VAL_FRAC)
    return set(prods[:nt]), set(prods[nt:nt + nv]), set(prods[nt + nv:])

a, b = split_products(spam_prods), split_products(other_prods)
test_p, val_p, train_p = a[0] | b[0], a[1] | b[1], a[2] | b[2]
assert not (test_p & train_p) and not (test_p & val_p) and not (val_p & train_p)

def assign(d):
    d = d.copy()
    d["split"] = np.select([d[prod].isin(test_p), d[prod].isin(val_p)], ["test", "val"], default="train")
    return d

real = assign(real)
spam = {k: assign(v) for k, v in spam.items()}

summary = {"real": real["split"].value_counts()}
for k, v in spam.items():
    summary[k] = v["split"].value_counts()
display(pd.DataFrame(summary).fillna(0).astype(int))

produktov so spamom: 500 | ostatných: 7094
produkty spoločné všetkým trom generátorom: 499


,real,llama3,qwen_72b,qwen_dsr1
split,,,,
test,8307,375,375,375
train,39270,1739,1745,1750
val,7990,375,375,375


## 4. Baseline: TF-IDF + logistická regresia
- TF-IDF: slovné 1–2-gramy, `min_df=3`, `sublinear_tf=True`
- logistická regresia, `class_weight="balanced"`
- train na jednom generátore (train produkty), test na každom generátore (test produkty)
- diagonála matice: ten istý generátor; mimo diagonály: neznámy generátor
- hlavná metrika: PR-AUC (nezávisí od prahu; accuracy pri podiele spamu ~4 % nie je informatívna); F1 pri prahu 0.5

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, average_precision_score, roc_auc_score

def build(name, split):
    r = real[real["split"] == split]
    s = spam[name][spam[name]["split"] == split]
    X = np.concatenate([r[txt].values, s[txt].values])
    y = np.r_[np.zeros(len(r)), np.ones(len(s))]
    return X, y

rows = []
for train_llm in spam:
    Xtr, ytr = build(train_llm, "train")
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True)
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(vec.fit_transform(Xtr), ytr)
    for test_llm in spam:
        Xte, yte = build(test_llm, "test")
        proba = clf.predict_proba(vec.transform(Xte))[:, 1]
        pr, rc, f1, _ = precision_recall_fscore_support(yte, proba >= 0.5, average="binary", zero_division=0)
        rows.append(dict(train=train_llm, test=test_llm, precision=pr, recall=rc, f1=f1,
                         pr_auc=average_precision_score(yte, proba), roc_auc=roc_auc_score(yte, proba),
                         base_rate=yte.mean(), n_test_spam=int(yte.sum()), n_test=len(yte)))

res = pd.DataFrame(rows).round(3)
display(res)
print("PR-AUC (riadky = train, stĺpce = test):")
display(res.pivot(index="train", columns="test", values="pr_auc"))
print("F1 pri prahu 0.5:")
display(res.pivot(index="train", columns="test", values="f1"))

,train,test,precision,recall,f1,pr_auc,roc_auc,base_rate,n_test_spam,n_test
0,llama3,llama3,0.947,0.992,0.969,0.995,1.000,0.043,375,8682
1,llama3,qwen_72b,0.932,0.763,0.839,0.937,0.996,0.043,375,8682
2,llama3,qwen_dsr1,0.940,0.883,0.911,0.969,0.998,0.043,375,8682
3,qwen_72b,llama3,0.906,0.877,0.892,0.949,0.997,0.043,375,8682
4,qwen_72b,qwen_72b,0.916,0.987,0.950,0.988,0.999,0.043,375,8682
5,qwen_72b,qwen_dsr1,0.908,0.896,0.902,0.957,0.997,0.043,375,8682
6,qwen_dsr1,llama3,0.870,0.571,0.689,0.836,0.988,0.043,375,8682
7,qwen_dsr1,qwen_72b,0.833,0.427,0.564,0.774,0.985,0.043,375,8682
8,qwen_dsr1,qwen_dsr1,0.920,0.979,0.948,0.993,1.000,0.043,375,8682


PR-AUC (riadky = train, stĺpce = test):


test,llama3,qwen_72b,qwen_dsr1
train,,,
llama3,0.995,0.937,0.969
qwen_72b,0.949,0.988,0.957
qwen_dsr1,0.836,0.774,0.993


F1 pri prahu 0.5:


test,llama3,qwen_72b,qwen_dsr1
train,,,
llama3,0.969,0.839,0.911
qwen_72b,0.892,0.950,0.902
qwen_dsr1,0.689,0.564,0.948


## 5. Test artefaktu z dĺžky
- spam má úzke rozdelenie dĺžky slov, pravé recenzie široké -> ROC-AUC z poradia podľa dĺžky signál podhodnocuje
- rozhodovací strom (`max_depth=4`) len z počtu slov zachytí aj pásmo dĺžok

In [10]:
from sklearn.tree import DecisionTreeClassifier

def wc(X):
    return np.array([len(str(t).split()) for t in X]).reshape(-1, 1)

first = list(spam)[0]
Xtr, ytr = build(first, "train")
Xte, yte = build(first, "test")
tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=SEED).fit(wc(Xtr), ytr)
p = tree.predict_proba(wc(Xte))[:, 1]
print(f"len-only strom ({first}): PR-AUC {average_precision_score(yte, p):.3f} | ROC-AUC {roc_auc_score(yte, p):.3f} | náhodný PR-AUC = {yte.mean():.3f}")

len-only strom (llama3): PR-AUC 0.189 | ROC-AUC 0.901 | náhodný PR-AUC = 0.043


In [11]:
Path("../../results").mkdir(exist_ok=True)
res.to_csv("../results/baseline_tfidf_lr.csv", index=False)
print("uložené: ../results/baseline_tfidf_lr.csv")

uložené: ../results/baseline_tfidf_lr.csv


## Výsledok
- čistenie: 86 758 -> 55 567 pravých recenzií
- metadáta spamu: `verified_purchase` = 0, `helpful_vote` = 0, chýba `reviewSummary`, `product_star` sedí so `star` len v ~27 %, `reviewDay` s `reviewTime` v ~16 % -> metadáta nepoužité ako vstup
- rozdelenie (seed 42): test 8 307 pravých + 375 spamu na generátor
- PR-AUC: diagonála 0,988–0,995; mimo diagonály 0,774–0,969; najslabší transfer z qwen_dsr1 (0,836 a 0,774)
- dĺžka (rozhodovací strom, llama3): PR-AUC 0,189 pri podiele spamu 0,043, ROC-AUC 0,901